In [27]:
import polars as pl
import polars.selectors as cs

In [28]:
df_prf = pl.read_csv("./data/anuario_prf.csv")

In [29]:
df_prf.head(10)

data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,veiculos,latitude,longitude,risco
str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,i64,i64,f64,f64,str
"""2022-01-01""","""sábado""","""02:40:00""","""PR""",116,33,"""CAMPINA GRANDE DO SUL""","""Ingestão de álcool pelo condut…","""Tombamento""","""Pleno dia""","""Decrescente""","""Nublado""","""Dupla""","""Curva""","""Não""",3,2,-25.114403,-48.846755,"""alto"""
"""2022-01-01""","""sábado""","""05:22:00""","""MS""",163,393,"""NOVA ALVORADA DO SUL""","""Condutor deixou de manter dist…","""Colisão traseira""","""Amanhecer""","""Decrescente""","""Céu Claro""","""Simples""","""Aclive""","""Não""",3,3,-21.228445,-54.456296,"""medio"""
"""2022-01-01""","""sábado""","""07:00:00""","""RJ""",101,457,"""ANGRA DOS REIS""","""Reação tardia ou ineficiente d…","""Colisão frontal""","""Pleno dia""","""Decrescente""","""Chuva""","""Simples""","""Curva""","""Sim""",2,2,-23.031498,-44.177153,"""alto"""
"""2022-01-01""","""sábado""","""09:00:00""","""MG""",40,508,"""RIBEIRAO DAS NEVES""","""Acumulo de água sobre o pavime…","""Saída de leito carroçável""","""Pleno dia""","""Decrescente""","""Chuva""","""Dupla""","""Reta""","""Sim""",3,1,-19.760612,-44.134754,"""baixo"""
"""2022-01-01""","""sábado""","""09:00:00""","""PB""",116,8,"""CACHOEIRA DOS INDIOS""","""Mal súbito do condutor""","""Colisão com objeto""","""Pleno dia""","""Crescente""","""Céu Claro""","""Simples""","""Reta""","""Não""",3,2,-6.964668,-38.727608,"""baixo"""
"""2022-01-01""","""sábado""","""10:20:00""","""MG""",40,452,"""CAETANOPOLIS""","""Chuva""","""Colisão lateral mesmo sentido""","""Pleno dia""","""Crescente""","""Chuva""","""Dupla""","""Declive""","""Não""",4,3,-19.333821,-44.361079,"""medio"""
"""2022-01-01""","""sábado""","""08:05:00""","""SC""",163,80,"""GUARACIABA""","""Ausência de reação do condutor""","""Colisão frontal""","""Pleno dia""","""Crescente""","""Céu Claro""","""Simples""","""Declive;Curva""","""Não""",4,3,-26.650439,-53.518546,"""alto"""
"""2022-01-01""","""sábado""","""12:20:00""","""SC""",101,139,"""BALNEARIO CAMBORIU""","""Condutor deixou de manter dist…","""Colisão traseira""","""Pleno dia""","""Crescente""","""Céu Claro""","""Simples""","""Aclive""","""Não""",5,4,-27.029823,-48.60115,"""medio"""
"""2022-01-01""","""sábado""","""14:15:00""","""CE""",116,415,"""IPAUMIRIM""","""Manobra de mudança de faixa""","""Colisão frontal""","""Pleno dia""","""Decrescente""","""Nublado""","""Simples""","""Reta""","""Não""",3,5,-6.741404,-38.792746,"""alto"""


In [30]:
df_prf.schema

Schema([('data_inversa', String),
        ('dia_semana', String),
        ('horario', String),
        ('uf', String),
        ('br', Int64),
        ('km', Int64),
        ('municipio', String),
        ('causa_acidente', String),
        ('tipo_acidente', String),
        ('fase_dia', String),
        ('sentido_via', String),
        ('condicao_metereologica', String),
        ('tipo_pista', String),
        ('tracado_via', String),
        ('uso_solo', String),
        ('pessoas', Int64),
        ('veiculos', Int64),
        ('latitude', Float64),
        ('longitude', Float64),
        ('risco', String)])

In [31]:
TARGET_COL = "risco"

INFERENCE_COL = "causa_acidente"

DROP_COLS = ["municipio", "dia_semana"]
NUMERIC_CLEANUP_COLS = ["km", "latitude", "longitude"]
DATE_TIME_ORIGINALS = ["data_inversa", "horario"]

In [32]:
cond_especial = (
    pl.col("tracado_via").str.contains("Ponte") |
    pl.col("tracado_via").str.contains("Viaduto") |
    pl.col("tracado_via").str.contains("Túnel")
)

cond_aclive_declive = (
    pl.col("tracado_via").str.contains("Aclive") |
    pl.col("tracado_via").str.contains("Declive")
)

In [33]:
df_clean = df_prf.with_columns(
    # pl.col(NUMERIC_CLEANUP_COLS).str.replace_all(",", ".").cast(pl.Float64).fill_null(0.0),
    pl.col("br").cast(pl.Utf8),
    pl.col("data_inversa").str.to_date(format="%Y-%m-%d", strict=False).alias("data"),
    pl.col("horario").str.to_time(format="%H:%M:%S", strict=False).alias("hora_limpa"),
).with_columns(
    pl.col("data").dt.month().alias("mes"),
    pl.col("data").dt.weekday().alias("dia_semana_num"),
    pl.col("hora_limpa").dt.hour().alias("hora"),
).with_columns(
    pl.col("tracado_via")
    .str.split(";")  
    .list.sort()       
    .list.join(";")
    .alias('tracado_via')
).with_columns(
   pl.when(pl.col("tracado_via").str.contains("Reta")).then(pl.lit("Reta"))
      .when(pl.col("tracado_via").str.contains("Curva")).then(pl.lit("Curva"))
      .when(pl.col("tracado_via").str.contains("Interseção")).then(pl.lit("Intersecao"))
      .when(cond_especial).then(pl.lit("Especial"))
      .when(cond_aclive_declive).then(pl.lit("Aclive_Declive"))
      .otherwise(pl.lit("Outro"))
      .alias('tracado_via')
).with_columns(
    cs.string().cast(pl.Categorical).fill_null("Desconhecido"),
    (cs.numeric() - cs.float()).fill_null(0),
)

In [34]:
df_prf.filter((pl.col("latitude").abs() > 90) | (pl.col("longitude").abs() > 180))

data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,veiculos,latitude,longitude,risco
str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,i64,i64,f64,f64,str


In [35]:
cols_to_drop_final = DROP_COLS + DATE_TIME_ORIGINALS + ["hora_limpa", "hora"]

In [36]:
df_final = df_clean.drop(cols_to_drop_final)

In [37]:
df_final

uf,br,km,causa_acidente,tipo_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,veiculos,latitude,longitude,risco,data,mes,dia_semana_num
cat,cat,i64,cat,cat,cat,cat,cat,cat,cat,cat,i64,i64,f64,f64,cat,date,i8,i8
"""PR""","""116""",33,"""Ingestão de álcool pelo condut…","""Tombamento""","""Pleno dia""","""Decrescente""","""Nublado""","""Dupla""","""Curva""","""Não""",3,2,-25.114403,-48.846755,"""alto""",2022-01-01,1,6
"""MS""","""163""",393,"""Condutor deixou de manter dist…","""Colisão traseira""","""Amanhecer""","""Decrescente""","""Céu Claro""","""Simples""","""Aclive_Declive""","""Não""",3,3,-21.228445,-54.456296,"""medio""",2022-01-01,1,6
"""RJ""","""101""",457,"""Reação tardia ou ineficiente d…","""Colisão frontal""","""Pleno dia""","""Decrescente""","""Chuva""","""Simples""","""Curva""","""Sim""",2,2,-23.031498,-44.177153,"""alto""",2022-01-01,1,6
"""MG""","""40""",508,"""Acumulo de água sobre o pavime…","""Saída de leito carroçável""","""Pleno dia""","""Decrescente""","""Chuva""","""Dupla""","""Reta""","""Sim""",3,1,-19.760612,-44.134754,"""baixo""",2022-01-01,1,6
"""PB""","""116""",8,"""Mal súbito do condutor""","""Colisão com objeto""","""Pleno dia""","""Crescente""","""Céu Claro""","""Simples""","""Reta""","""Não""",3,2,-6.964668,-38.727608,"""baixo""",2022-01-01,1,6
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SC""","""282""",302,"""Velocidade Incompatível""","""Saída de leito carroçável""","""Pleno dia""","""Crescente""","""Céu Claro""","""Simples""","""Reta""","""Não""",1,1,-27.519538,-50.909615,"""medio""",2025-09-27,9,6
"""SP""","""381""",82,"""Ausência de reação do condutor""","""Colisão com objeto""","""Pleno dia""","""Crescente""","""Céu Claro""","""Múltipla""","""Reta""","""Sim""",2,1,-23.481235,-46.562748,"""medio""",2025-08-30,8,6
"""RJ""","""493""",20,"""Ausência de reação do condutor""","""Colisão com objeto""","""Pleno dia""","""Crescente""","""Sol""","""Simples""","""Curva""","""Não""",1,1,-22.660571,-43.046321,"""medio""",2025-08-25,8,1


In [38]:
df_final["risco"].value_counts()

risco,count
cat,u32
"""baixo""",42432
"""alto""",73277
"""medio""",143028


In [39]:
df_final.write_parquet("./data/anuario_prf.parquet")